# ℹ️ This copy runs on its own

**Nothing is saved to your Google Drive.** This notebook downloads the demo files it needs into the Colab session's temporary disk, shows the results, and leaves nothing behind: when you close the notebook, the downloaded files are gone with the session.

You only need to be signed in to a Google account to run it in Colab. Select `Runtime` -> `Run all`.

---



# Sine-wave demo: what the attention branches learn

This notebook shows the attention of DiverScan on a synthetic dataset, where what the model should find is known in advance.

Each recording is a 10-second sine wave whose frequency switches every 0.5 to 1 second. Under the `SingleFreq` condition the frequency stays between 6 and 9 Hz. Under the `MixedFreq` condition, short segments at 1 Hz and at 14 Hz are added. A model with two attention branches was trained to tell the two conditions apart, five times with different random seeds. Every cell below reads the finished analysis from the demo project, downloaded into this session:

        DiverScan-release
        │
        └── projects
                 └── sinewave_demo
                          ├── dataset     the recordings (the test set is what is shown here)
                          └── results     the model's predictions and attention on the test set, one folder per seed


In [ ]:
#@title ⬇️ Download the demo files into this session
#@markdown Nothing is written to your Google Drive. The files are downloaded into this Colab session's temporary disk and disappear when the session ends.

import subprocess
import sys
import zipfile
from pathlib import Path

# DiverScan-release.zip in the shared Google Drive folder; only the parts of
# the archive listed in wanted_prefixes are extracted.
archive_id = "1T2c3p_3-ofSY8zf8pNpes1oac6PhOiwR"
wanted_prefixes = ["projects/sinewave_demo/results/", "projects/sinewave_demo/dataset/whitenoise_before_normalization.npz", "projects/sinewave_demo/dataset/clean_freq_ts.npz"]

work_folder = Path("/content") if Path("/content").is_dir() else Path.cwd()
archive_path = work_folder / "DiverScan-release.zip"
project_folder = work_folder / "DiverScan-release" / "projects" / "sinewave_demo"

if not project_folder.is_dir():
    if not archive_path.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        gdown.download(id=archive_id, output=str(archive_path), quiet=False)
    with zipfile.ZipFile(archive_path) as archive:
        wanted = [m for m in archive.namelist() if m.startswith(tuple(wanted_prefixes))]
        archive.extractall(work_folder / "DiverScan-release", members=wanted)
    archive_path.unlink()  # the archive is no longer needed
print("demo files:", project_folder)


# ⚙️ Load the demo project and look at the attention

1.   Select `Runtime` -> `Run all`.

2.   Pick a seed, a condition and a recording in the first cell. In every seed, one branch settles on the 1 Hz segments and the other on the 14 Hz segments.


In [ ]:
#@title Visualization of attention
#@markdown Attention of the two branches of the DiverScan model on the sine-wave dataset (MixedFreq vs. SingleFreq), for each of the five training seeds.
#@markdown Pick a seed, a condition and a file. The plot shows the signal, the frequency of the sine wave over time, and the attention of Branch 1 and Branch 2 (STFT frames, 100 per second).
#@markdown Red bands mark the 1 Hz segments and green bands the 14 Hz segments; only MixedFreq recordings have them.

import gzip
import pickle
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not in Colab

# The demo project, downloaded into this session by the cell above.
project_folder = Path(project_folder)
dataset_folder = project_folder / "dataset"
results_folder = project_folder / "results"

# The recordings (test set) and the frequency of the sine wave at every sample.
signals = np.load(dataset_folder / "whitenoise_before_normalization.npz", allow_pickle=True)
frequencies = np.load(dataset_folder / "clean_freq_ts.npz", allow_pickle=True)

sampling_rate = 100.0
test_signal = signals["test_signal"][:, 0, :]        # (files, samples)
test_frequency = frequencies["test_freq_ts"]         # (files, samples), Hz
test_mask_low = signals["test_mask_low"]             # (files, samples), 1 inside 1 Hz segments
test_mask_high = signals["test_mask_high"]           # (files, samples), 1 inside 14 Hz segments
signal_time = np.arange(test_signal.shape[1]) / sampling_rate
attention_time = signals["stft_times"]               # (frames,), seconds
branch_titles = ["Attention (DiverScan, Branch 1)", "Attention (DiverScan, Branch 2)"]

# One folder per seed under results/, e.g. seed45_epoch23, holding the
# attention the model gave to every STFT frame of every test file.
seed_names = sorted(p.name for p in results_folder.iterdir() if p.is_dir())
visualizations = {}


def load_visualization(seed_name):
    if seed_name not in visualizations:
        with gzip.open(results_folder / seed_name / "dictionary_for_visualization.pkl", "rb") as f:
            visualizations[seed_name] = pickle.load(f)
    return visualizations[seed_name]


class_name_list = list(load_visualization(seed_names[0]))


def file_names(seed_name, condition):
    return [str(f) for f in load_visualization(seed_name)[condition]]


def sample_index(file_name):
    """The test-set index of a file: 'MixedFreq/sample_0166.npz' -> 166."""
    return int(re.search(r"sample_(\d+)", file_name).group(1))


def shade_segments(ax, time_axis, mask, color):
    """Shade every run of ones in mask."""
    indices = np.where(mask > 0.5)[0]
    if len(indices) == 0:
        return
    step = time_axis[1] - time_axis[0]
    starts = [indices[0]] + [i for previous, i in zip(indices, indices[1:]) if i != previous + 1]
    ends = [previous for previous, i in zip(indices, indices[1:]) if i != previous + 1] + [indices[-1]]
    for start, end in zip(starts, ends):
        ax.axvspan(time_axis[start], time_axis[end] + step, color=color, alpha=0.2, linewidth=0)


def plot_attention(seed_name, condition, file_name):
    sample = load_visualization(seed_name)[condition][file_name]
    i = sample_index(file_name)
    attentions = sample["attentions"][:, 0, :]       # (branches, frames)
    predicted = [class_name_list[p] for p in sample["prediction"]]

    print("Selected:")
    print("    ⭐️", seed_name, "/", condition, "/", file_name)
    for branch, name in enumerate(predicted):
        print(f"    Branch {branch + 1} predicted {name}" + ("" if name == condition else " (wrong)"))

    fig, axes = plt.subplots(2 + len(attentions), 1, figsize=(12, 10), gridspec_kw=dict(hspace=0.6))

    ax = axes[0]
    ax.plot(signal_time, test_signal[i], color="black", linewidth=0.8)
    shade_segments(ax, signal_time, test_mask_low[i], "tab:red")
    shade_segments(ax, signal_time, test_mask_high[i], "tab:green")
    ax.set_title(condition)
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, 10)

    ax = axes[1]
    ax.step(signal_time, test_frequency[i], where="post", color="black", linewidth=1.2)
    shade_segments(ax, signal_time, test_mask_low[i], "tab:red")
    shade_segments(ax, signal_time, test_mask_high[i], "tab:green")
    ax.set_title("Frequency")
    ax.set_ylabel("Hz")
    ax.set_yticks([1, 7, 14])
    ax.set_ylim(-0.5, 15.5)
    ax.set_xlim(0, 10)

    for branch, attention in enumerate(attentions):
        ax = axes[2 + branch]
        ax.plot(attention_time, attention, color="magenta", linewidth=1.2)
        ax.fill_between(attention_time, 0, attention, color="magenta", alpha=0.25)
        shade_segments(ax, attention_time, test_mask_low[i], "tab:red")
        shade_segments(ax, attention_time, test_mask_high[i], "tab:green")
        ax.set_title(branch_titles[branch] if branch < len(branch_titles) else f"Attention (DiverScan, Branch {branch + 1})")
        ax.set_ylabel("Value")
        ax.set_ylim(0, 0.02)
        ax.set_yticks([0, 0.01, 0.02])
        ax.set_xlim(0, 10)
    axes[-1].set_xlabel("Time (s)")

    plt.show()
    plt.close(fig)


# Dropdowns: the seed, the condition, and the files of that condition.
seed_dropdown = widgets.Dropdown(options=seed_names, description="Seed")
condition_dropdown = widgets.Dropdown(options=class_name_list, description="Condition")
file_dropdown = widgets.Dropdown(options=file_names(seed_names[0], class_name_list[0]), description="Filename")
plot_output = widgets.Output()


def update_files(change):
    file_dropdown.options = file_names(seed_dropdown.value, condition_dropdown.value)


def update_plot(change):
    with plot_output:
        clear_output(wait=True)
        plot_attention(seed_dropdown.value, condition_dropdown.value, file_dropdown.value)


seed_dropdown.observe(update_files, names="value")
condition_dropdown.observe(update_files, names="value")
seed_dropdown.observe(update_plot, names="value")
condition_dropdown.observe(update_plot, names="value")
file_dropdown.observe(update_plot, names="value")

display(widgets.VBox([widgets.HBox([seed_dropdown, condition_dropdown, file_dropdown]), plot_output]))
update_plot(None)


In [ ]:
#@title Model prediction of all files in the test set
#@markdown "True" means that the branch predicted the condition of the file correctly, "False" that it did not. Run the cell above first.

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

prediction_tables = {}


def load_prediction_table(seed_name):
    if seed_name not in prediction_tables:
        tables = pd.read_excel(results_folder / seed_name / "prediction_true_false.xlsx", sheet_name=None, index_col=0)
        for table in tables.values():
            table.columns = [f"Branch {h + 1}" for h in range(table.shape[1])]
        prediction_tables[seed_name] = tables
    return prediction_tables[seed_name]


table_seed_dropdown = widgets.Dropdown(options=seed_names, description="Seed")
table_condition_dropdown = widgets.Dropdown(options=class_name_list, description="Condition")
table_output = widgets.Output()


def update_table(change):
    with table_output:
        clear_output(wait=True)
        table = load_prediction_table(table_seed_dropdown.value)[table_condition_dropdown.value]
        print("Selected:")
        print("    ⭐️", table_seed_dropdown.value, "/", table_condition_dropdown.value,
              f"({table.shape[0]} files; correct: " + ", ".join(f"{c} {table[c].mean():.1%}" for c in table.columns) + ")")
        display(table)


table_seed_dropdown.observe(update_table, names="value")
table_condition_dropdown.observe(update_table, names="value")
display(widgets.VBox([widgets.HBox([table_seed_dropdown, table_condition_dropdown]), table_output]))
update_table(None)
